In [ ]:
import os
from pathlib import Path
import pandas as pd
import csv

# =========================
# CONFIGURAÇÃO
# =========================
PASTA = 
ARQUIVO_SAIDA = 
ARQUIVO_LOG = 

COL_DATA = "DATA_ABERTURA"
COL_CHAVE = "NUM_SGO"

# =========================
# FUNÇÕES AUXILIARES
# =========================
def detectar_delimitador(arquivo, enc="utf-8-sig", amostra_bytes=20000):
    """Tenta detectar delimitador automaticamente (típico: ; , \t)."""
    try:
        with open(arquivo, "r", encoding=enc, errors="ignore") as f:
            sample = f.read(amostra_bytes)
        sniffer = csv.Sniffer()
        dialect = sniffer.sniff(sample, delimiters=[",", ";", "\t", "|"])
        return dialect.delimiter
    except Exception:
        # fallback: tab (muito comum em export de sistemas) ou ;
        return "\t"

def ler_csv_robusto(arquivo):
    """
    Lê CSV tentando:
    - detectar delimitador
    - testar encodings comuns
    - manter tudo como texto (dtype=str) para evitar notação científica em IDs/CPFs
    """
    encodings = ["utf-8-sig", "utf-8", "latin1", "cp1252"]
    last_err = None

    for enc in encodings:
        try:
            delim = detectar_delimitador(arquivo, enc=enc)
            df = pd.read_csv(
                arquivo,
                sep=delim,
                dtype=str,
                engine="python",
                encoding=enc,
                on_bad_lines="skip"
            )
            return df, enc, delim
        except Exception as e:
            last_err = e

    raise RuntimeError(f"Falha ao ler {arquivo}. Último erro: {last_err}")

def normalizar_data_abertura(df):
    """
    Converte DATA_ABERTURA para datetime robusto.
    Trata:
    - Texto (DD/MM/YYYY, YYYY-MM-DD, etc.)
    - Números (Excel serial)
    """

    if COL_DATA not in df.columns:
        return df

    # Normaliza texto
    df[COL_DATA] = df[COL_DATA].astype(str).str.strip()

    # --- 1. Tenta converter direto (texto comum)
    dt = pd.to_datetime(df[COL_DATA], errors="coerce", dayfirst=True)

    # --- 2. Trata possíveis números (Excel)
    # Identifica valores numéricos válidos
    mask_numerico = df[COL_DATA].str.match(r'^\d+$')

    if mask_numerico.any():
        numeros = pd.to_numeric(df.loc[mask_numerico, COL_DATA], errors="coerce")

        # Converte número Excel -> datetime
        dt_excel = pd.to_datetime(numeros, origin="1899-12-30", unit="D", errors="coerce")

        # Substitui onde conversão padrão falhou
        dt.loc[mask_numerico] = dt.loc[mask_numerico].fillna(dt_excel)

    # --- 3. Formata saída final (importantíssimo para Celonis)
    df[COL_DATA] = dt.dt.strftime("%Y-%m-%d %H:%M:%S")
    
    # mantém para ordenação e lógica interna
    df["_DATA_ABERTURA_DT"] = dt

    return df

# =========================
# PROCESSAMENTO PRINCIPAL
# =========================
def main():
    pasta = Path(PASTA)
    if not pasta.exists():
        raise FileNotFoundError(f"Pasta não encontrada: {PASTA}")

    arquivos = sorted(pasta.glob("*.csv"))
    if not arquivos:
        print(f"Nenhum CSV encontrado em: {PASTA}")
        return

    relatorio = []
    dfs = []

    print("=" * 80)
    print("CONSOLIDAÇÃO SGO - INÍCIO")
    print(f"Pasta: {PASTA}")
    print(f"Arquivos CSV encontrados: {len(arquivos)}")
    print("=" * 80)

    for arq in arquivos:
        df, enc, delim = ler_csv_robusto(arq)
        linhas = len(df)

        relatorio.append({
            "arquivo": arq.name,
            "linhas": linhas,
            "encoding": enc,
            "delimitador": repr(delim)
        })

        print(f"[OK] {arq.name} | linhas={linhas} | enc={enc} | sep={repr(delim)}")

        dfs.append(df)

    # Consolida
    df_all = pd.concat(dfs, ignore_index=True)

    # Garante que chave exista
    if COL_CHAVE not in df_all.columns:
        raise KeyError(f"Coluna obrigatória não encontrada: {COL_CHAVE}")

    # Limpa chave (NUM_SGO) para evitar duplicidade por espaços
    df_all[COL_CHAVE] = df_all[COL_CHAVE].astype(str).str.strip()

    # Normaliza data
    df_all = normalizar_data_abertura(df_all)

    if "_DATA_ABERTURA_DT" not in df_all.columns:
        # Se não existe coluna de data, não dá para aplicar "mais recente"
        raise KeyError(f"Coluna obrigatória não encontrada: {COL_DATA}")

    total_antes = len(df_all)

    # 1) Ordena por DATA_ABERTURA (conforme pedido)
    # (NaT vai para o fim; mas vamos tratar para manter datas válidas como "mais recentes")
    df_all["_DATA_VALIDA"] = df_all["_DATA_ABERTURA_DT"].notna()
    df_all = df_all.sort_values(
        by=["_DATA_VALIDA", "_DATA_ABERTURA_DT"],
        ascending=[True, True],
        kind="mergesort"  # estável
    )

    # 2) Remove duplicados por NUM_SGO mantendo a data mais recente:
    # como ordenamos crescente, "keep='last'" mantém o mais recente (maior data),
    # e também privilegia registros com data válida sobre NaT.
    df_dedup = df_all.drop_duplicates(subset=[COL_CHAVE], keep="last")

    total_depois = len(df_dedup)
    excluidos_dup = total_antes - total_depois

    # 3) Ordena o resultado final por DATA_ABERTURA (deixando mais antigo -> mais novo)
    df_dedup = df_dedup.sort_values(by=["_DATA_ABERTURA_DT"], ascending=True, kind="mergesort")

    # Remove colunas auxiliares
    df_dedup = df_dedup.drop(columns=["_DATA_ABERTURA_DT", "_DATA_VALIDA"], errors="ignore")

    # Salva na mesma pasta
    saida_path = pasta / ARQUIVO_SAIDA
    df_dedup.to_csv(saida_path, index=False, encoding="utf-8-sig")

    # Gera LOG
    log_path = pasta / ARQUIVO_LOG
    with open(log_path, "w", encoding="utf-8") as f:
        f.write("CONSOLIDAÇÃO SGO - RESUMO\n")
        f.write(f"Pasta: {PASTA}\n")
        f.write(f"Arquivos processados: {len(arquivos)}\n\n")

        f.write("ARQUIVOS:\n")
        for r in relatorio:
            f.write(f"- {r['arquivo']} | linhas={r['linhas']} | enc={r['encoding']} | sep={r['delimitador']}\n")

        f.write("\nTOTAIS:\n")
        f.write(f"- Linhas antes (consolidadas): {total_antes}\n")
        f.write(f"- Linhas depois (deduplicadas): {total_depois}\n")
        f.write(f"- Linhas excluídas por duplicidade (NUM_SGO + mais recente por DATA_ABERTURA): {excluidos_dup}\n")
        f.write(f"\nArquivo gerado: {saida_path.name}\n")

    print("=" * 80)
    print("CONSOLIDAÇÃO SGO - FIM")
    print(f"Linhas antes (consolidadas): {total_antes}")
    print(f"Linhas depois (deduplicadas): {total_depois}")
    print(f"Linhas excluídas por duplicidade: {excluidos_dup}")
    print(f"Arquivo gerado: {saida_path}")
    print(f"LOG gerado: {log_path}")
    print("=" * 80)

if __name__ == "__main__":
    main()